# OI Directional — Evaluation (notebooks_8/05)

Standalone model — no MFE filter.
Trades whenever OI directional model confidence exceeds threshold.

In [ ]:
import pandas as pd
import numpy as np
import joblib
import warnings
import gc
warnings.filterwarnings('ignore')
from pathlib import Path

FEAT6_DIR  = Path('../backend/data/features_6')
FEAT8_DIR  = Path('../backend/data/features_8')
PRICE_DIR  = Path('../backend/data/processed')
MODELS_DIR = Path('../backend/models_9/oi_directional')

MAJORS    = ['EURUSD', 'GBPUSD', 'USDJPY', 'USDCHF', 'USDCAD', 'AUDUSD', 'NZDUSD']
TRAIN_END = '2024-06-30'

SPREADS = {
    'EURUSD': 0.2 * 0.0001,
    'GBPUSD': 0.4 * 0.0001,
    'USDJPY': 0.3 * 0.01 / 150,
    'USDCHF': 0.4 * 0.0001,
    'USDCAD': 0.4 * 0.0001,
    'AUDUSD': 0.3 * 0.0001,
    'NZDUSD': 0.5 * 0.0001,
}

print('Ready.')

## 1. Load Model

In [ ]:
bundle       = joblib.load(MODELS_DIR / 'model_oi_directional.joblib')
oi_feat_cols = bundle['feature_cols']
models       = bundle['models']

print(f'Features: {len(oi_feat_cols)}')
print(f'CV AUC:')
for k, v in bundle['cv_auc'].items():
    print(f'  {k}: {v:.4f}')


## 2. Load Test Data & Predict

In [ ]:
all_bars = []

for pair in MAJORS:
    print(f'  {pair}...', flush=True)

    df6 = pd.read_parquet(FEAT6_DIR / f'{pair}_features.parquet')
    df8 = pd.read_parquet(FEAT8_DIR / f'{pair}_geometric.parquet')

    df6.drop(columns=['pair'], errors='ignore', inplace=True)
    df8.drop(columns=['pair'], errors='ignore', inplace=True)

    idx = df6.index.intersection(df8.index)
    df  = pd.concat([df6.loc[idx], df8.loc[idx]], axis=1)
    df  = df.loc[:, ~df.columns.duplicated()]
    del df6, df8; gc.collect()

    df = df[df.index > TRAIN_END]

    price_df = pd.read_parquet(PRICE_DIR / f'{pair}_1H.parquet')
    close = price_df['close'].reindex(df.index)
    del price_df

    num_cols = [c for c in df.columns if df[c].dtype != object]
    df[num_cols] = df[num_cols].ffill().fillna(0).astype(np.float32)

    X = np.zeros((len(df), len(oi_feat_cols)), dtype=np.float32)
    for i, c in enumerate(oi_feat_cols):
        if c in df.columns:
            X[:, i] = df[c].values

    rows = pd.DataFrame(index=df.index)
    for name in ['oi_4H', 'oi_12H']:
        probs = models[name].predict_proba(X)
        rows[f'{name}_down'] = probs[:, 0]
        rows[f'{name}_flat'] = probs[:, 1]
        rows[f'{name}_up']   = probs[:, 2]
    del X; gc.collect()

    rows['pair']   = pair
    rows['close']  = close.values
    rows['spread'] = SPREADS[pair]
    all_bars.append(rows)
    del df, rows; gc.collect()

bars = pd.concat(all_bars).sort_index(); del all_bars; gc.collect()
nm = (bars.index.max() - bars.index.min()).days / 30
print(f'\nTest bars: {len(bars):,}  ({nm:.1f} months)')

## 3. Threshold Sweep

In [ ]:
def simulate(bars, horizon_col, thresh, nm):
    h = int(horizon_col.split('_')[1].replace('H',''))
    sim_rows = []
    for pair in sorted(bars['pair'].unique()):
        df_pair = bars[bars['pair'] == pair].sort_index()
        spread  = df_pair['spread'].iloc[0]
        in_trade_until = pd.Timestamp.min
        for k in range(len(df_pair)):
            ts  = df_pair.index[k]
            if ts < in_trade_until: continue
            row = df_pair.iloc[k]
            p_up   = row[f'{horizon_col}_up']
            p_down = row[f'{horizon_col}_down']
            if p_up > thresh and p_up > p_down:
                direction = 'long'
            elif p_down > thresh and p_down > p_up:
                direction = 'short'
            else:
                continue
            exit_ts    = ts + pd.Timedelta(hours=h)
            future     = df_pair[df_pair.index >= exit_ts]
            exit_close = future['close'].iloc[0] if len(future) > 0 else df_pair['close'].iloc[-1]
            log_ret    = np.log(exit_close / row['close'])
            pnl        = (log_ret if direction == 'long' else -log_ret) - spread
            sim_rows.append({'pnl': pnl, 'direction': direction, 'pair': pair, 'ts': ts})
            in_trade_until = exit_ts
    if not sim_rows: return None
    s = pd.DataFrame(sim_rows)
    pnl = s['pnl']
    ev = pnl.mean(); wr = (pnl > 0).mean()
    sh = (ev / pnl.std()) * np.sqrt(252*24) if pnl.std() > 0 else 0
    return {'n': len(s), 'wr': wr, 'ev': ev, 'sh': sh, 'pmo': pnl.sum()/nm, 'df': s}

print(f'{"Horizon":<10} {"Thresh":>7} {"N":>7} {"N/mo":>6} {"WR":>7} {"EV":>10} {"Sharpe":>8} {"PnL/mo":>10}')
print('=' * 78)
sweep_results = {}
for horizon_col in ['oi_4H', 'oi_12H']:
    for thresh in [0.35, 0.40, 0.45, 0.50, 0.55, 0.60]:
        r = simulate(bars, horizon_col, thresh, nm)
        if r is None: continue
        sweep_results[(horizon_col, thresh)] = r
        flag = ' <<<' if r['ev'] > 0 else ''
        print(f'{horizon_col:<10} {thresh:>7.2f} {r["n"]:>7,} {r["n"]/nm:>6.1f} {r["wr"]:>7.1%} {r["ev"]:>+10.6f} {r["sh"]:>+8.2f} {r["pmo"]:>+10.4f}{flag}')
    print()


## 4. Best Config Deep Dive

In [ ]:
# Pick best by Sharpe among positive EV configs
best_key = max(
    [(k, v) for k, v in sweep_results.items() if v['ev'] > 0],
    key=lambda x: x[1]['sh']
)[0]
BEST_HORIZON, BEST_THRESH = best_key
best = sweep_results[best_key]
print(f'Best config: {BEST_HORIZON}  thresh={BEST_THRESH}')
print(f'  N={best["n"]:,}  WR={best["wr"]:.1%}  EV={best["ev"]:+.6f}  Sharpe={best["sh"]:+.2f}  PnL/mo={best["pmo"]:+.4f}')

sub = best['df'].set_index('ts')
pnl = sub['pnl']

print(f'\nPER-PAIR:')
print(f'  {"Pair":<10} {"N":>6} {"/mo":>5} {"WR":>7} {"EV":>10} {"PnL/mo":>10}')
print('  ' + '-'*52)
for pair, g in sub.groupby('pair'):
    wr  = (g['pnl']>0).mean(); ev = g['pnl'].mean()
    pmo = g['pnl'].sum()/nm;   npm = len(g)/nm
    flag = ' <<<' if ev > 0 else ''
    print(f'  {pair:<10} {len(g):>6,} {npm:>5.1f} {wr:>7.1%} {ev:>+10.6f} {pmo:>+10.4f}{flag}')

print(f'\nBY DIRECTION:')
for d, g in sub.groupby('direction'):
    wr = (g['pnl']>0).mean(); ev = g['pnl'].mean()
    print(f'  {d:<6} N={len(g):,}  WR={wr:.1%}  EV={ev:+.6f}  PnL/mo={g["pnl"].sum()/nm:+.4f}')

print(f'\nMONTHLY:')
print(f'  {"Month":<10} {"N":>5} {"WR":>7} {"EV":>10} {"CumPnL":>10}')
print('  ' + '-'*48)
cum = 0
for (yr, mo), g in sub.groupby([sub.index.year, sub.index.month]):
    ev = g['pnl'].mean(); cum += g['pnl'].sum()
    wr = (g['pnl']>0).mean()
    flag = ' <--' if ev < 0 else ''
    print(f'  {yr}-{mo:02d}    {len(g):>5} {wr:>7.1%} {ev:>+10.6f} {cum:>+10.4f}{flag}')
